# Task 4: Open-set recognition
Known classes: CIFAR-10. Unknowns: 16 fixed CIFAR-100 test classes, used only in the final evaluation section

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import copy
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10, CIFAR100
from torchvision.models import resnet18

CFG = {
    "seed": 6304,
    "val_fraction": 0.1,                 # stratified 90/10 split of the official CIFAR-10 train set
    "batch_size": 128,
    "epochs": 100, "lr": 0.1, "momentum": 0.9, "weight_decay": 5e-4,      # vanilla and gcsc
    "randaugment": {"num_ops": 2, "magnitude": 9},                         # gcsc only
    "proser": {"epochs": 50, "lr": 1e-3, "n_dummy": 5, "beta": 1.0, "gamma": 0.1, "mixup_alpha": 2.0},
    "accept_rate": 0.95,                 # threshold = 95th percentile of unknownness on CIFAR-10 val
    "mahalanobis_eps": 1e-6,
    "near": ["bus", "pickup_truck", "motorcycle", "tractor", "wolf", "fox", "leopard", "camel"],
    "far": ["bottle", "bowl", "chair", "clock", "keyboard", "mushroom", "sunflower", "wardrobe"],
    "amp": True,                         # mixed precision on the gpu, about twice as fast on a T4
    "num_workers": 2,
}
RUNS = {"vanilla": {"augment": "standard", "epochs": CFG["epochs"], "lr": CFG["lr"]},
        "gcsc": {"augment": "randaugment", "epochs": CFG["epochs"], "lr": CFG["lr"]},
        "proser": {"augment": "standard", "epochs": CFG["proser"]["epochs"], "lr": CFG["proser"]["lr"],
                   "init_from": "vanilla"}}

REPO = Path("/content/drive/MyDrive/atml-pa1")
RESULTS = REPO / "task4" / "results"
FIGS = RESULTS / "figures"
LOGS = RESULTS / "logs"
CACHE = Path("/content/drive/MyDrive/pa1-data/task4")     # checkpoints and saved features/logits
CKPT = CACHE / "checkpoints"
DATA_ROOT = "/content/data"                                 # datasets on the fast local disk
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [3]:
if not REPO.exists():
    !git clone https://github.com/mardyweb/atml-pa1.git {REPO}

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_json(path):
    with open(path) as f:
        return json.load(f)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)


def save_table(df, name):
    df.to_csv(RESULTS / f"{name}.csv", index=False)
    try:
        df.to_latex(RESULTS / f"{name}.tex", index=False, float_format="%.3f", escape=True)
    except Exception:
        pass
    display(df.round(3))


for folder in [RESULTS, FIGS, LOGS, CACHE, CKPT]:
    folder.mkdir(parents=True, exist_ok=True)
set_seed(CFG["seed"])
save_json({"config": CFG, "runs": RUNS, "device": str(DEVICE),
           "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__,
                        "sklearn": sklearn.__version__, "numpy": np.__version__}}, RESULTS / "run_info.json")

# ---- same look as the other tasks ----
SCORE_COLORS = {"MSP": "#2E4057", "MLS": "#00798C", "Energy": "#EDAE49", "Mahalanobis": "#D1495B"}
MODEL_COLORS = {"vanilla": "#2E4057", "gcsc": "#00798C", "proser": "#D1495B", "proser_ph": "#EDAE49"}
MODEL_LABELS = {"vanilla": "Vanilla", "gcsc": "GCSC", "proser": "PROSER", "proser_ph": "PROSER (placeholder)"}
GROUP_COLORS = {"known": "#8AB17D", "near": "#D1495B", "far": "#EDAE49"}
INK = "#2B2B2B"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 8.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#E4E0D6", "grid.linewidth": 0.7,
    "legend.frameon": False, "legend.fontsize": 8,
    "figure.dpi": 110, "savefig.bbox": "tight", "pdf.fonttype": 42,
})


def save_fig(fig, name):
    fig.savefig(FIGS / f"{name}.pdf")
    fig.savefig(FIGS / f"{name}.png", dpi=300)
    plt.show()
    plt.close(fig)

## Data

In [5]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
EVAL_TF = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])


def train_transform(kind):
    """crop + flip, and for gcsc RandAugment in between the flip and the conversion."""
    ops = [T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()]
    if kind == "randaugment":
        ops.append(T.RandAugment(**CFG["randaugment"]))
    return T.Compose(ops + [T.ToTensor(), T.Normalize(MEAN, STD)])


# the official train set, once with and once without augmentation (same images, same order)
base_train = CIFAR10(DATA_ROOT, train=True, download=True)
CLASS_NAMES = list(base_train.classes)
N_KNOWN = len(CLASS_NAMES)
targets = np.array(base_train.targets)

# stratified 90/10 split, saved once
if (RESULTS / "splits.json").exists():
    SPLITS = load_json(RESULTS / "splits.json")
else:
    tr, va = train_test_split(np.arange(len(targets)), test_size=CFG["val_fraction"], stratify=targets,
                              random_state=CFG["seed"])
    SPLITS = {"seed": CFG["seed"], "train_idx": sorted(tr.tolist()), "val_idx": sorted(va.tolist())}
    save_json(SPLITS, RESULTS / "splits.json")
TRAIN_IDX, VAL_IDX = SPLITS["train_idx"], SPLITS["val_idx"]

# evaluation-only sets: unaugmented train, val, test, and the two CIFAR-100 unknown groups
EVAL_SETS = {
    "train": Subset(CIFAR10(DATA_ROOT, train=True, transform=EVAL_TF), TRAIN_IDX),
    "val": Subset(CIFAR10(DATA_ROOT, train=True, transform=EVAL_TF), VAL_IDX),
    "test": CIFAR10(DATA_ROOT, train=False, download=True, transform=EVAL_TF),
}
c100 = CIFAR100(DATA_ROOT, train=False, download=True, transform=EVAL_TF)
C100_NAMES = list(c100.classes)
c100_targets = np.array(c100.targets)
for group in ["near", "far"]:
    ids = [c100.class_to_idx[name] for name in CFG[group]]
    EVAL_SETS[group] = Subset(c100, np.where(np.isin(c100_targets, ids))[0].tolist())
print({k: len(v) for k, v in EVAL_SETS.items()})
assert len(EVAL_SETS["near"]) == 800 and len(EVAL_SETS["far"]) == 800


def eval_loader(name):
    return DataLoader(EVAL_SETS[name], batch_size=512, shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)

100%|██████████| 170M/170M [33:06<00:00, 85.8kB/s]
100%|██████████| 169M/169M [32:31<00:00, 86.6kB/s]


{'train': 45000, 'val': 5000, 'test': 10000, 'near': 800, 'far': 800}


## Model: CIFAR ResNet-18

In [6]:
class CifarResNet18(nn.Module):
    """torchvision ResNet-18 with a 3x3 stride-1 stem and no max-pool, for 32x32 inputs.
    The network is cut after layer2 so that PROSER can mix features there."""

    def __init__(self, n_dummy=0):
        super().__init__()
        net = resnet18(weights=None, num_classes=N_KNOWN)
        net.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        net.maxpool = nn.Identity()
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.layer1, net.layer2)   # up to layer2
        self.tail = nn.Sequential(net.layer3, net.layer4, net.avgpool, nn.Flatten())     # layer3 to feature
        self.fc = net.fc                                                                  # 10 known classes
        self.dummy = nn.Linear(512, n_dummy) if n_dummy else None                        # PROSER placeholders

    def pre(self, x):
        return self.stem(x)

    def post(self, h):
        return self.tail(h)              # 512-d penultimate feature

    def heads(self, feat):
        return self.fc(feat), (self.dummy(feat) if self.dummy is not None else None)

    def forward(self, x):
        feat = self.post(self.pre(x))
        logits, dummy = self.heads(feat)
        return feat, logits, dummy


def augmented_logits(logits, dummy):
    """PROSER's K+1 output: the ten known logits and the strongest dummy response."""
    return torch.cat([logits, dummy.max(dim=1, keepdim=True).values], dim=1).float()

## Training (one loop for Vanilla, GCSC and PROSER)

In [7]:
def proser_loss(model, x, y, pc):
    """Classifier placeholders on the first half of the batch, data placeholders on the second half.
    Losses follow Zhou et al. (2021): the dummy class is index K in the augmented output."""
    half = len(x) // 2
    dummy_target = torch.full((half,), N_KNOWN, device=x.device, dtype=torch.long)

    # --- classifier placeholders: true class first, and with the true class removed the dummy class first
    feat, logits, dummy = model(x[:half])
    aug = augmented_logits(logits, dummy)                                    # (half, K+1)
    loss_known = F.cross_entropy(aug, y[:half])
    masked = aug.clone()
    masked[torch.arange(half, device=x.device), y[:half]] = -1e9              # remove the ground-truth logit
    loss_cls_ph = loss_known + pc["beta"] * F.cross_entropy(masked, dummy_target)

    # --- data placeholders: manifold mixup after layer2 between examples of different classes
    xb, yb = x[half:], y[half:]
    h = model.pre(xb)
    perm = torch.randperm(len(xb), device=x.device)
    keep = yb != yb[perm]                                                     # only pairs from different classes
    lam = torch.distributions.Beta(pc["mixup_alpha"], pc["mixup_alpha"]).sample().item()
    n_pairs = int(keep.sum())
    if n_pairs == 0:                                                          # every label identical, no valid pair
        loss_data_ph = torch.zeros((), device=x.device)
    else:
        h_mix = lam * h[keep] + (1 - lam) * h[perm][keep]
        feat_mix = model.post(h_mix)
        logits_mix, dummy_mix = model.heads(feat_mix)
        loss_data_ph = F.cross_entropy(augmented_logits(logits_mix, dummy_mix),
                                       torch.full((n_pairs,), N_KNOWN, device=x.device, dtype=torch.long))

    return loss_cls_ph + pc["gamma"] * loss_data_ph, {"cls_ph": loss_cls_ph.item(), "data_ph": loss_data_ph.item()}


@torch.no_grad()
def val_accuracy(model):
    """Closed-set accuracy on the CIFAR-10 validation split, known logits only."""
    model.eval()
    correct = 0
    for x, y in eval_loader("val"):
        _, logits, _ = model(x.to(DEVICE, non_blocking=True))
        correct += (logits.argmax(1).cpu() == y).sum().item()
    return correct / len(EVAL_SETS["val"])


def train_run(run):
    spec = RUNS[run]
    best_path, last_path, log_path = CKPT / f"{run}_best.pt", CKPT / f"{run}_last.pt", LOGS / f"{run}.json"
    if best_path.exists() and log_path.exists():
        print(f"{run}: already trained, skipping")
        return

    set_seed(CFG["seed"])
    is_proser = run == "proser"
    model = CifarResNet18(n_dummy=CFG["proser"]["n_dummy"] if is_proser else 0)
    if is_proser:
        # start from the selected vanilla checkpoint, only the five dummy classifiers are new
        state = torch.load(CKPT / f"{spec['init_from']}_best.pt", map_location="cpu")["model"]
        missing, unexpected = model.load_state_dict(state, strict=False)
        assert not unexpected and all(k.startswith("dummy.") for k in missing), (missing, unexpected)
    model = model.to(DEVICE)

    opt = torch.optim.SGD(model.parameters(), lr=spec["lr"], momentum=CFG["momentum"],
                          weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=spec["epochs"])
    use_amp = CFG["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=use_amp)

    train_set = Subset(CIFAR10(DATA_ROOT, train=True, transform=train_transform(spec["augment"])), TRAIN_IDX)
    loader_gen = torch.Generator()
    start_epoch, best_acc, best_epoch, best_state, log = 1, -1.0, 0, None, []

    # pick up where a disconnected session left off
    if last_path.exists():
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        start_epoch, best_acc, best_epoch, best_state, log = ck["epoch"] + 1, ck["best_acc"], ck["best_epoch"], ck["best_state"], ck["log"]
        print(f"{run}: resuming from epoch {start_epoch}")

    for epoch in range(start_epoch, spec["epochs"] + 1):
        t0 = time.time()
        loader_gen.manual_seed(CFG["seed"] * 1000 + epoch)          # same batch order for this epoch, resume or not
        loader = DataLoader(train_set, batch_size=CFG["batch_size"], shuffle=True, drop_last=True,
                            num_workers=CFG["num_workers"], pin_memory=True, generator=loader_gen)
        model.train()
        sums, n = {"loss": 0.0, "cls_ph": 0.0, "data_ph": 0.0}, 0
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                if is_proser:
                    loss, parts = proser_loss(model, x, y, CFG["proser"])
                    sums["cls_ph"] += parts["cls_ph"]
                    sums["data_ph"] += parts["data_ph"]
                else:
                    _, logits, _ = model(x)
                    loss = F.cross_entropy(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sums["loss"] += loss.item()
            n += 1
        sched.step()

        acc = val_accuracy(model)
        entry = {"epoch": epoch, "train_loss": sums["loss"] / n, "val_acc": acc, "lr": opt.param_groups[0]["lr"]}
        if is_proser:
            entry.update({"cls_placeholder_loss": sums["cls_ph"] / n, "data_placeholder_loss": sums["data_ph"] / n})
        log.append(entry)
        if acc > best_acc:
            best_acc, best_epoch = acc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"{run} | epoch {epoch:3d} | loss {entry['train_loss']:.4f} | val acc {acc:.4f} | best {best_acc:.4f} (ep {best_epoch}) | {time.time() - t0:.0f}s")

        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch, "best_acc": best_acc, "best_epoch": best_epoch,
                    "best_state": best_state, "log": log}, last_path)

    torch.save({"model": best_state, "best_epoch": best_epoch, "best_val_acc": best_acc, "spec": spec}, best_path)
    save_json({"run": run, "spec": spec, "best_epoch": best_epoch, "best_val_acc": best_acc, "log": log}, log_path)
    last_path.unlink(missing_ok=True)
    print(f"{run}: kept epoch {best_epoch} (val acc {best_acc:.4f})")

## Step 1: Vanilla closed-set baseline

In [8]:
train_run("vanilla")

vanilla | epoch   1 | loss 2.0079 | val acc 0.3832 | best 0.3832 (ep 1) | 24s
vanilla | epoch   2 | loss 1.5154 | val acc 0.4662 | best 0.4662 (ep 2) | 24s
vanilla | epoch   3 | loss 1.2630 | val acc 0.5550 | best 0.5550 (ep 3) | 26s
vanilla | epoch   4 | loss 1.0488 | val acc 0.6432 | best 0.6432 (ep 4) | 26s
vanilla | epoch   5 | loss 0.8969 | val acc 0.6812 | best 0.6812 (ep 5) | 26s
vanilla | epoch   6 | loss 0.7831 | val acc 0.7240 | best 0.7240 (ep 6) | 27s
vanilla | epoch   7 | loss 0.6882 | val acc 0.7646 | best 0.7646 (ep 7) | 27s
vanilla | epoch   8 | loss 0.6178 | val acc 0.7004 | best 0.7646 (ep 7) | 27s
vanilla | epoch   9 | loss 0.5791 | val acc 0.7514 | best 0.7646 (ep 7) | 28s
vanilla | epoch  10 | loss 0.5451 | val acc 0.7538 | best 0.7646 (ep 7) | 28s
vanilla | epoch  11 | loss 0.5217 | val acc 0.7564 | best 0.7646 (ep 7) | 28s
vanilla | epoch  12 | loss 0.5020 | val acc 0.8020 | best 0.8020 (ep 12) | 28s
vanilla | epoch  13 | loss 0.4756 | val acc 0.8110 | best 0.811

## Step 3: GCSC (vanilla recipe + RandAugment)

In [9]:
train_run("gcsc")

gcsc | epoch   1 | loss 2.2039 | val acc 0.3440 | best 0.3440 (ep 1) | 46s
gcsc | epoch   2 | loss 1.6855 | val acc 0.4162 | best 0.4162 (ep 2) | 40s
gcsc | epoch   3 | loss 1.3982 | val acc 0.5108 | best 0.5108 (ep 3) | 43s
gcsc | epoch   4 | loss 1.1228 | val acc 0.6300 | best 0.6300 (ep 4) | 43s
gcsc | epoch   5 | loss 0.9398 | val acc 0.7022 | best 0.7022 (ep 5) | 41s
gcsc | epoch   6 | loss 0.8254 | val acc 0.7132 | best 0.7132 (ep 6) | 41s
gcsc | epoch   7 | loss 0.7664 | val acc 0.7432 | best 0.7432 (ep 7) | 41s
gcsc | epoch   8 | loss 0.7157 | val acc 0.7452 | best 0.7452 (ep 8) | 42s
gcsc | epoch   9 | loss 0.6848 | val acc 0.7878 | best 0.7878 (ep 9) | 41s
gcsc | epoch  10 | loss 0.6641 | val acc 0.7320 | best 0.7878 (ep 9) | 40s
gcsc | epoch  11 | loss 0.6420 | val acc 0.7980 | best 0.7980 (ep 11) | 41s
gcsc | epoch  12 | loss 0.6253 | val acc 0.8252 | best 0.8252 (ep 12) | 43s
gcsc | epoch  13 | loss 0.6001 | val acc 0.7886 | best 0.8252 (ep 12) | 43s
gcsc | epoch  14 | los

## Step 4: PROSER (fine-tuned from the vanilla checkpoint)

In [10]:
train_run("proser")

proser | epoch   1 | loss 0.6203 | val acc 0.9470 | best 0.9470 (ep 1) | 31s
proser | epoch   2 | loss 0.3529 | val acc 0.9478 | best 0.9478 (ep 2) | 28s
proser | epoch   3 | loss 0.3361 | val acc 0.9450 | best 0.9478 (ep 2) | 30s
proser | epoch   4 | loss 0.3272 | val acc 0.9450 | best 0.9478 (ep 2) | 31s
proser | epoch   5 | loss 0.3087 | val acc 0.9470 | best 0.9478 (ep 2) | 32s
proser | epoch   6 | loss 0.2916 | val acc 0.9464 | best 0.9478 (ep 2) | 31s
proser | epoch   7 | loss 0.2006 | val acc 0.9420 | best 0.9478 (ep 2) | 31s
proser | epoch   8 | loss 0.1102 | val acc 0.9366 | best 0.9478 (ep 2) | 32s
proser | epoch   9 | loss 0.1026 | val acc 0.9378 | best 0.9478 (ep 2) | 31s
proser | epoch  10 | loss 0.0894 | val acc 0.9362 | best 0.9478 (ep 2) | 31s
proser | epoch  11 | loss 0.1063 | val acc 0.9348 | best 0.9478 (ep 2) | 32s
proser | epoch  12 | loss 0.0837 | val acc 0.9352 | best 0.9478 (ep 2) | 31s
proser | epoch  13 | loss 0.0490 | val acc 0.9012 | best 0.9478 (ep 2) | 32s

## Extract features and logits
All checkpoints are fixed at this point. Every score below reads the same saved outputs.

In [ ]:
missing = [r for r in RUNS if not (CKPT / f"{r}_best.pt").exists()]
assert not missing, f"train these first: {missing}"
save_json({"locked_at": time.strftime("%Y-%m-%d %H:%M:%S"),
           "selection_rule": "highest CIFAR-10 validation accuracy",
           "threshold_rule": f"{int(100 * CFG['accept_rate'])}th percentile of unknownness on the CIFAR-10 validation split",
           "checkpoints": {r: {k: load_json(LOGS / f"{r}.json")[k] for k in ("best_epoch", "best_val_acc")} for r in RUNS}},
          RESULTS / "locked_before_unknown_eval.json")


def load_model(run):
    model = CifarResNet18(n_dummy=CFG["proser"]["n_dummy"] if run == "proser" else 0)
    model.load_state_dict(torch.load(CKPT / f"{run}_best.pt", map_location="cpu")["model"])
    return model.to(DEVICE).eval()


@torch.no_grad()
def extract(model, name):
    feats, logits, dummies, labels = [], [], [], []
    for x, y in eval_loader(name):
        f, z, d = model(x.to(DEVICE, non_blocking=True))
        feats.append(f.float().cpu())
        logits.append(z.float().cpu())
        labels.append(y)
        if d is not None:
            dummies.append(d.float().cpu())
    out = {"feat": torch.cat(feats), "logit": torch.cat(logits), "label": torch.cat(labels)}
    if dummies:
        out["dummy"] = torch.cat(dummies)
    return out


OUT = {}
for run in RUNS:
    path = CACHE / f"outputs_{run}.pt"
    if path.exists():
        OUT[run] = torch.load(path)
    else:
        model = load_model(run)
        OUT[run] = {name: extract(model, name) for name in EVAL_SETS}
        torch.save(OUT[run], path)
        del model
    print(run, {k: tuple(v["feat"].shape) for k, v in OUT[run].items()})

# closed-set accuracy on the full CIFAR-10 test set, known logits only
CSA = {run: float((OUT[run]["test"]["logit"].argmax(1) == OUT[run]["test"]["label"]).float().mean()) for run in RUNS}
print({k: round(v, 4) for k, v in CSA.items()})

vanilla {'train': (45000, 512), 'val': (5000, 512), 'test': (10000, 512), 'near': (800, 512), 'far': (800, 512)}
gcsc {'train': (45000, 512), 'val': (5000, 512), 'test': (10000, 512), 'near': (800, 512), 'far': (800, 512)}


## Step 2: Post-hoc unknownness scores (larger = more novel)

In [ ]:
def score_msp(o, stats):
    return (1 - o["logit"].softmax(1).max(1).values).numpy()


def score_mls(o, stats):
    return (-o["logit"].max(1).values).numpy()


def score_energy(o, stats):
    return (-torch.logsumexp(o["logit"], dim=1)).numpy()


def score_mahalanobis(o, stats):
    """Smallest squared Mahalanobis distance to any known-class mean, shared diagonal covariance."""
    f = o["feat"]
    d = torch.stack([(((f - mu) ** 2) / stats["var"]).sum(1) for mu in stats["means"]], dim=1)
    return d.min(1).values.numpy()


def score_placeholder(o, stats):
    """PROSER's detection score, as in the reference code: softmax over the K+1 augmented logits,
    unknownness = dummy probability minus the largest known-class probability."""
    p = augmented_logits(o["logit"], o["dummy"]).softmax(1)
    return (p[:, N_KNOWN] - p[:, :N_KNOWN].max(1).values).numpy()


SCORES = {"MSP": score_msp, "MLS": score_mls, "Energy": score_energy, "Mahalanobis": score_mahalanobis}


def mahalanobis_stats(train_out):
    """Class means and one shared diagonal covariance from unaugmented training features."""
    f, y = train_out["feat"], train_out["label"]
    means = torch.stack([f[y == c].mean(0) for c in range(N_KNOWN)])
    resid = f - means[y]
    var = resid.var(0, unbiased=False) + CFG["mahalanobis_eps"]
    return {"means": means, "var": var}


# unknownness of every evaluated example: U[run][score][split]
U = {}
for run in RUNS:
    stats = mahalanobis_stats(OUT[run]["train"])
    U[run] = {s: {split: fn(OUT[run][split], stats) for split in ["val", "test", "near", "far"]}
              for s, fn in SCORES.items()}
    if run == "proser":
        U[run]["Placeholder"] = {split: score_placeholder(OUT[run][split], None) for split in ["val", "test", "near", "far"]}
torch.save(U, CACHE / "unknownness_scores.pt")

## Step 6: Common evaluation

In [ ]:
def osr_metrics(u):
    """AUROC over all thresholds, plus behaviour at the validation-calibrated threshold."""
    tau = float(np.percentile(u["val"], 100 * CFG["accept_rate"]))          # accept if u <= tau
    known, near, far = u["test"], u["near"], u["far"]
    auroc = lambda unk: float(roc_auc_score(np.r_[np.zeros(len(known)), np.ones(len(unk))], np.r_[known, unk]))
    return {"AUROC near": auroc(near), "AUROC far": auroc(far), "AUROC all": auroc(np.r_[near, far]),
            "test acceptance": float((known <= tau).mean()),
            "near rejection": float((near > tau).mean()), "far rejection": float((far > tau).mean()),
            "FPR95 near": float((near <= tau).mean()), "FPR95 far": float((far <= tau).mean()), "threshold": tau}


METRICS = {run: {s: osr_metrics(u) for s, u in U[run].items()} for run in RUNS}
save_json({"csa": CSA, "metrics": METRICS}, RESULTS / "osr_metrics.json")

# table 1: the four post-hoc scores on the frozen vanilla model
rows = [{"score": s, **{k: v for k, v in METRICS["vanilla"][s].items() if k != "threshold"}} for s in SCORES]
save_table(pd.DataFrame(rows), "table_vanilla_scores")

# table 2: trained models with MLS, plus PROSER with its placeholder score
COMPARE = [("vanilla", "MLS"), ("gcsc", "MLS"), ("proser", "MLS"), ("proser", "Placeholder")]
ROW_KEY = {("vanilla", "MLS"): "vanilla", ("gcsc", "MLS"): "gcsc", ("proser", "MLS"): "proser",
           ("proser", "Placeholder"): "proser_ph"}
rows = []
for run, s in COMPARE:
    m = METRICS[run][s]
    rows.append({"model": MODEL_LABELS[ROW_KEY[(run, s)]], "score": s, "CSA": CSA[run],
                 **{k: m[k] for k in ["AUROC near", "AUROC far", "AUROC all", "test acceptance",
                                      "near rejection", "far rejection"]}})
save_table(pd.DataFrame(rows), "table_model_comparison")

In [ ]:
# which unknown classes slip through, and which CIFAR-10 label absorbs them
near_lab, far_lab = OUT["vanilla"]["near"]["label"].numpy(), OUT["vanilla"]["far"]["label"].numpy()
rows = []
for group, labs in [("near", near_lab), ("far", far_lab)]:
    for name in CFG[group]:
        sel = labs == c100.class_to_idx[name]
        row = {"group": group, "unknown class": name, "n": int(sel.sum())}
        # acceptance rate of this class under every score of the vanilla model
        for s in SCORES:
            m = METRICS["vanilla"][s]
            row[f"acc. {s}"] = float((U["vanilla"][s][group][sel] <= m["threshold"]).mean())
        # and under the trained-model comparison
        for run, s in COMPARE[1:]:
            row[f"acc. {MODEL_LABELS[ROW_KEY[(run, s)]]}"] = float((U[run][s][group][sel] <= METRICS[run][s]["threshold"]).mean())
        # label the vanilla model gives the accepted ones (MLS threshold)
        accepted = sel & (U["vanilla"]["MLS"][group] <= METRICS["vanilla"]["MLS"]["threshold"])
        preds = OUT["vanilla"][group]["logit"][torch.from_numpy(accepted)].argmax(1).numpy()
        if len(preds):
            top = np.bincount(preds, minlength=N_KNOWN)
            row["absorbed by"], row["share"] = CLASS_NAMES[int(top.argmax())], float(top.max() / len(preds))
        else:
            row["absorbed by"], row["share"] = "-", 0.0
        rows.append(row)
UNKNOWN_CLASSES = pd.DataFrame(rows)
save_table(UNKNOWN_CLASSES, "table_unknown_classes")

# do the four scores rank the same examples as novel? spearman over test + all unknowns
allu = {s: np.r_[U["vanilla"][s]["test"], U["vanilla"][s]["near"], U["vanilla"][s]["far"]] for s in SCORES}
corr = pd.DataFrame([[spearmanr(allu[a], allu[b]).correlation for b in SCORES] for a in SCORES],
                    index=list(SCORES), columns=list(SCORES)).reset_index().rename(columns={"index": "score"})
save_table(corr, "table_score_rank_correlation")

# disagreements at the operating point: unknowns accepted by one score but rejected by the other
rows = []
for a in SCORES:
    for b in SCORES:
        if a >= b:
            continue
        ua, ub = np.r_[U["vanilla"][a]["near"], U["vanilla"][a]["far"]], np.r_[U["vanilla"][b]["near"], U["vanilla"][b]["far"]]
        acc_a, acc_b = ua <= METRICS["vanilla"][a]["threshold"], ub <= METRICS["vanilla"][b]["threshold"]
        rows.append({"score A": a, "score B": b, "A accepts, B rejects": float((acc_a & ~acc_b).mean()),
                     "B accepts, A rejects": float((~acc_a & acc_b).mean()), "both accept": float((acc_a & acc_b).mean())})
save_table(pd.DataFrame(rows), "table_score_disagreement")

## Figures

In [ ]:
# score distributions and ROC curves for MSP, MLS and Mahalanobis on the vanilla model
shown = ["MSP", "MLS", "Mahalanobis"]
fig, axes = plt.subplots(2, 3, figsize=(7.0, 4.2))
for j, s in enumerate(shown):
    u, m = U["vanilla"][s], METRICS["vanilla"][s]
    ax = axes[0, j]
    lo = min(u["test"].min(), u["near"].min(), u["far"].min())
    hi = np.percentile(np.r_[u["test"], u["near"], u["far"]], 99.5)
    bins = np.linspace(lo, hi, 40)
    for group, key in [("known", "test"), ("near", "near"), ("far", "far")]:
        ax.hist(np.clip(u[key], lo, hi), bins=bins, density=True, alpha=0.55, color=GROUP_COLORS[group],
                label=f"{group} (CIFAR-10 test)" if group == "known" else f"{group} unknown")
    ax.axvline(m["threshold"], color=INK, linestyle="--", linewidth=1)
    ax.set_title(f"{s}: unknownness", color=SCORE_COLORS[s], fontweight="bold", fontsize=9)
    ax.set_yticks([])
    ax.grid(axis="y", visible=False)

    ax = axes[1, j]
    for group, ls in [("near", "-"), ("far", "--")]:
        yy = np.r_[np.zeros(len(u["test"])), np.ones(len(u[group]))]
        fpr, tpr, _ = roc_curve(yy, np.r_[u["test"], u[group]])
        ax.plot(fpr, tpr, color=GROUP_COLORS[group], linestyle=ls, linewidth=1.6,
                label=f"{group} (AUROC {m[f'AUROC {group}']:.3f})")
    ax.plot([0, 1], [0, 1], color="#BBBBBB", linewidth=0.8)
    ax.set_xlabel("unknowns accepted (FPR)")
    if j == 0:
        ax.set_ylabel("unknowns rejected (TPR)")
    ax.legend(loc="lower right", fontsize=7)
axes[0, 0].legend(loc="upper right", fontsize=7)
axes[0, 0].set_ylabel("density")
fig.tight_layout(h_pad=0.8, w_pad=0.6)
save_fig(fig, "fig_score_distributions_roc")

In [ ]:
# unknowns that the vanilla model accepts under the MLS threshold, most confident first
tau = METRICS["vanilla"]["MLS"]["threshold"]
raw_c100 = CIFAR100(DATA_ROOT, train=False)          # unnormalised pixels for display
records = []
fig, axes = plt.subplots(2, 4, figsize=(7.0, 4.0))
for r, group in enumerate(["near", "far"]):
    u, labs = U["vanilla"]["MLS"][group], OUT["vanilla"][group]["label"].numpy()
    order = np.argsort(u)                             # lowest unknownness = accepted with the most confidence
    accepted = [int(i) for i in order if u[i] <= tau]
    # the most confidently accepted example of each unknown class, then the four most confident of those
    first_of_class = {}
    for i in accepted:
        first_of_class.setdefault(int(labs[i]), i)
    idx_in_group = sorted(first_of_class.values(), key=lambda i: u[i])[:4]
    subset_idx = np.array(EVAL_SETS[group].indices)
    for i in accepted[:10]:                           # the ten most confidently accepted, for the write-up
        records.append({"group": group, "unknown_class": C100_NAMES[int(labs[i])],
                        "predicted": CLASS_NAMES[int(OUT["vanilla"][group]["logit"][i].argmax())],
                        "score_mls": float(u[i]), "threshold": tau, "shown_in_figure": i in idx_in_group})
    for ax, i in zip(axes[r], idx_in_group):
        img = raw_c100.data[subset_idx[i]]
        pred = int(OUT["vanilla"][group]["logit"][i].argmax())
        true = C100_NAMES[int(OUT["vanilla"][group]["label"][i])]
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"{group}: {true}", fontsize=8, color=GROUP_COLORS[group], fontweight="bold")
        ax.text(0.5, -0.07, f"predicted {CLASS_NAMES[pred]}\nMLS {u[i]:.2f} (tau {tau:.2f})", transform=ax.transAxes,
                ha="center", va="top", fontsize=7)
fig.subplots_adjust(hspace=0.75, wspace=0.15)
save_fig(fig, "fig_accepted_unknowns")
save_json(records, RESULTS / "accepted_unknown_examples.json")

In [ ]:
# training curves for the three runs
logs = {r: load_json(LOGS / f"{r}.json") for r in RUNS}
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.2))
for run in ["vanilla", "gcsc"]:
    ep = [e["epoch"] for e in logs[run]["log"]]
    axes[0].plot(ep, [e["train_loss"] for e in logs[run]["log"]], color=MODEL_COLORS[run], linewidth=1.4, label=MODEL_LABELS[run])
    axes[1].plot(ep, [100 * e["val_acc"] for e in logs[run]["log"]], color=MODEL_COLORS[run], linewidth=1.4, label=MODEL_LABELS[run])
ep = [e["epoch"] for e in logs["proser"]["log"]]
axes[2].plot(ep, [e["cls_placeholder_loss"] for e in logs["proser"]["log"]], color=MODEL_COLORS["proser"], linewidth=1.4, label="classifier placeholder")
axes[2].plot(ep, [e["data_placeholder_loss"] for e in logs["proser"]["log"]], color=MODEL_COLORS["proser_ph"], linewidth=1.4, label="data placeholder")
ax2 = axes[2].twinx()
ax2.plot(ep, [100 * e["val_acc"] for e in logs["proser"]["log"]], color="#8A8578", linewidth=1.0, linestyle=":")
ax2.set_ylabel("val acc (%)", fontsize=7, color="#8A8578")
ax2.grid(False)
for ax, title in zip(axes, ["train loss (Vanilla, GCSC)", "CIFAR-10 val accuracy (%)", "PROSER fine-tuning losses"]):
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("epoch")
    ax.legend(fontsize=7)
axes[1].set_ylim(bottom=max(0, axes[1].get_ylim()[0]))
fig.tight_layout(w_pad=1.2)
save_fig(fig, "fig_training_curves")

In [ ]:
# near vs far: ranking quality and rejection at the calibrated threshold, for the model comparison
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.3))
width, x = 0.2, np.arange(2)
for j, (run, s) in enumerate(COMPARE):
    key, m = ROW_KEY[(run, s)], METRICS[run][s]
    for ax, cols, title in [(axes[0], ["AUROC near", "AUROC far"], "AUROC"),
                            (axes[1], ["near rejection", "far rejection"], f"unknowns rejected at {int(100 * CFG['accept_rate'])}% known acceptance")]:
        vals = [100 * m[c] for c in cols]
        bars = ax.bar(x + (j - 1.5) * width, vals, width, color=MODEL_COLORS[key], edgecolor="white", linewidth=0.6,
                      label=f"{MODEL_LABELS[key]} (CSA {100 * CSA[run]:.1f}%)")
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.0f}", (b.get_x() + b.get_width() / 2, v), xytext=(0, 2), textcoords="offset points",
                        ha="center", fontsize=6, color=INK)
        ax.set_title(title, fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(["near unknowns", "far unknowns"])
        ax.set_ylim(0, 108)
        ax.grid(axis="x", visible=False)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.1), fontsize=7)
fig.tight_layout()
save_fig(fig, "fig_model_comparison")

## Commit to GitHub

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
%cd {REPO}
!git config user.name "mardyweb"
!git config user.email "maryamw17@outlook.com"
!git remote set-url origin https://{token}@github.com/mardyweb/atml-pa1.git
!git add task4
!git commit -m "Task 4: notebook, results and figures"
!git push